I want to compare all of the models we train here.

We will fit on a unified split using each model's best hyperparameters, this gives us the best fair comparison .

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score
import plotly.graph_objects as go

In [2]:
df = pd.read_csv("../data/drug_discovery_virtual_screening_cleaned.csv")
df = df.drop(columns=["Unnamed: 0"])


X = df.drop(columns=["active"])
y = df["active"]

X_train, X_test, y_train, y_test = train_test_split(
  X, y, test_size=0.2, stratify=y, random_state=42
)

In [3]:
scaler = StandardScaler().set_output(transform="pandas")
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 1599
Testing samples: 400


In [ ]:
# we need to specify for each whether or not it should be fit on scaled or unscaled features. 
models = {
    "Dummy (baseline)":    (DummyClassifier(strategy="most_frequent"), "scaled"),
    "KNN (k=41, cosine)":  (KNeighborsClassifier(n_neighbors=41, metric="cosine"), "scaled"),
    "Decision Tree (d=1)": (DecisionTreeClassifier(max_depth=1, random_state=42), "unscaled"),
    "Random Forest":       (RandomForestClassifier(n_estimators=100, max_depth=None, min_samples_split=2, random_state=42), "unscaled"),
    "Gradient Boosting":   (GradientBoostingClassifier(learning_rate=0.01, max_depth=1, n_estimators=44, random_state=42), "unscaled"),
    "LogReg L2 (C=10)":    (LogisticRegression(penalty="l2", solver="liblinear", C=10, max_iter=10000), "scaled"),
    "LogReg L1 (C=0.01)":  (LogisticRegression(penalty="l1", solver="liblinear", C=0.01, max_iter=10000), "scaled"),
}

In [18]:
# Here we fit each model 
results = []
fitted = {}

for name, (model, scaling) in models.items():
    if scaling == "scaled":
        Xtrain, Xtest = X_train_scaled, X_test_scaled
    else:
        Xtrain, Xtest = X_train, X_test

    model.fit(Xtrain, y_train)
    y_pred = model.predict(Xtest)
    y_prob = model.predict_proba(Xtest)[:, 1]

    auc = roc_auc_score(y_test, y_prob)
    acc = accuracy_score(y_test, y_pred)
    fpr, tpr, _ = roc_curve(y_test, y_prob)

    results.append({"model": name, "accuracy": acc, "roc_auc": auc})
    fitted[name] = {"fpr": fpr, "tpr": tpr, "auc": auc}

results_df = pd.DataFrame(results).sort_values("roc_auc", ascending=False).reset_index(drop=True)
results_df

,model,accuracy,roc_auc
0,LogReg L2 (C=10),0.8900,0.958043
1,LogReg L1 (C=0.01),0.8900,0.954476
2,Random Forest,0.8850,0.948947
3,"KNN (k=41, cosine)",0.8825,0.936372
4,Gradient Boosting,0.8925,0.869383
5,Decision Tree (d=1),0.8925,0.849068
6,Dummy (baseline),0.6950,0.500000


## ROC curve overlay

In [22]:
fig = go.Figure()
for name in results_df["model"]:
    info = fitted[name]
    label = f"{name} (AUC = {info['auc']:.3f})"
    fig.add_trace(go.Scatter(x=info["fpr"], y=info["tpr"], mode="lines", name=label))

fig.update_layout(
    title="ROC Curves For All Models",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate (Recall)",
    width=800,
    height=700
)
fig.show()